In [4]:
import pandas as pd

file_path = "inputs.xlsx"
df = pd.read_excel(file_path)
df

,link,id,fundingSource,institute,isHumanStudy,consentDescription,element_1A
0,https://grants.nih.gov/sites/default/files/flm...,1,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,"Demographic, clinical, and MRI, 1 H fMRS and f..."
1,https://grants.nih.gov/sites/default/files/flm...,2,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,Our genomic study will be registered with dbGa...
2,https://grants.nih.gov/sites/default/files/flm...,3,NIH,National Institute of Mental Health (NIMH),no,no,"As detailed in the Research Strategy Section, ..."
3,https://grants.nih.gov/sites/default/files/flm...,4,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,The data to be shared will include MRI images ...


In [ ]:
def build_prompt(row):
    funding_source = str(row.get("fundingSource", "Not provided")).strip()
    institute = str(row.get("institute", "Not provided")).strip()
    is_human = str(row.get("isHumanStudy", "No")).strip()
    consent = str(row.get("consentDescription", "no")).strip()
    data_collected = str(row.get("element_1A", "Not provided")).strip()

    consent_block = ""
    if is_human.lower() in ["yes", "true", "1"]:
        consent_block = f"\n- If yes, data sharing consent: {consent if consent else 'Not provided'}"

    prompt = f"""
You are a FAIR data expert helping researchers make their data more Findable, Accessible, Interoperable, and Reusable (FAIR).

Your task is to generate practical FAIR data instructions for a grant proposal being submitted to {funding_source}, specifically targeting {institute}.

Proposal context:
- Funding source: {funding_source}
- Human subjects study: {is_human}{consent_block}
- Data to be collected: {data_collected}

Based on these inputs, provide clear, practical, and user-friendly FAIR data guidance organized under the following sections:

1. Recommended file formats  
Explain which file format(s) should be used for this type of data and why they are appropriate for FAIR sharing and reuse.

2. Relevant tools and software  
List tools, software, or platforms that can help prepare, convert, validate, organize, document, or share this type of data.

3. Recommended dataset structure or standard  
State the dataset structure, schema, or community standard that should be followed, if one exists.

4. Dataset organization details  
Provide detailed guidance on how the dataset should be organized, including folder structure, file naming conventions, table organization, metadata arrangement, versioning considerations, and any other important technical details.

5. Metadata and documentation files to include  
List the metadata files, README files, data dictionaries, codebooks, protocols, or other documentation that should accompany the dataset.

6. Tools for metadata preparation and validation  
Suggest tools that can help create, validate, manage, or standardize metadata for this dataset.

7. De-identification and privacy considerations  
State whether de-identification is required. Explain what should be done based on the subject type, the data modality, and whether the study involves human participants. If consent language affects sharing, explain that clearly.

8. Recommended repositories  
Suggest appropriate repositories for sharing the data, prioritizing repositories recommended, required, or commonly accepted by the funding source and data type.

9. Recommended data licenses  
Recommend appropriate data license(s), prioritizing those expected, suggested, or required by the funding source. If restrictions apply because of human subjects or consent limitations, explain that clearly.

Instructions for your response:
- Make the guidance practical, concise, and easy for researchers to follow.
- Tailor all recommendations to the combination of funding source, human-subject status, consent language, and data modality.
- When funder policies, privacy, or consent restrictions affect the answer, explain that explicitly.
- When no single standard or repository is certain, label the answer as “Recommended approach” and provide the best evidence-based guidance.
- Prefer community-accepted standards, open formats, and reusable metadata practices whenever possible.
- Keep the tone professional, helpful, and grant-writing appropriate.
"""
    return prompt.strip()


# Create prompt column
df["prompt"] = df.apply(build_prompt, axis=1)

# Show first prompt
print(df["prompt"].iloc[0])

You are a FAIR data expert helping researchers make their data more Findable, Accessible, Interoperable, and Reusable (FAIR).

Your task is to generate practical FAIR data instructions for a grant proposal being submitted to NIH, specifically targeting National Institute of Mental Health (NIMH).

Proposal context:
- Funding source: NIH
- Human subjects study: yes
- If yes, data sharing consent: All research participants will be consented for broad data sharing.
- Data to be collected: Demographic, clinical, and MRI, 1 H fMRS and fMRI imaging data will be acquired from 110 affected youth and 110 matched healthy controls (described in detail in sections C.3 and C.4 of this application). All data will be deidentified prior to receipt by the repository, but the information needed to generate a global unique identifier for the NIMH Data Archive (NDA) will be collected for each subject.

Based on these inputs, provide clear, practical, and user-friendly FAIR data guidance organized under the

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

def ask_gpt(prompt_text):
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {
                "role": "system",
                "content": "You are a FAIR data expert helping generate practical FAIR instructions."
            },
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        temperature=0
    )
    return response.choices[0].message.content

# Apply to each row
df["gpt_answer"] = df["prompt"].apply(ask_gpt)

# Save output
df.to_excel("FAIR_instructions_with_prompts_and_answers1.xlsx", index=False)

In [7]:
for i, row in df.iterrows():
    print(f"\n--- Row {i} ---")
    print("PROMPT:\n", row["prompt"])
    print("\nANSWER:\n", row["gpt_answer"])
    print("\n" + "="*60)


--- Row 0 ---
PROMPT:
 You are a FAIR data expert helping researchers make their data more Findable, Accessible, Interoperable, and Reusable (FAIR).

Your task is to generate practical FAIR data instructions for a grant proposal being submitted to NIH, specifically targeting National Institute of Mental Health (NIMH).

Proposal context:
- Funding source: NIH
- Human subjects study: yes
- If yes, data sharing consent: All research participants will be consented for broad data sharing.
- Data to be collected: Demographic, clinical, and MRI, 1 H fMRS and fMRI imaging data will be acquired from 110 affected youth and 110 matched healthy controls (described in detail in sections C.3 and C.4 of this application). All data will be deidentified prior to receipt by the repository, but the information needed to generate a global unique identifier for the NIMH Data Archive (NDA) will be collected for each subject.

Based on these inputs, provide clear, practical, and user-friendly FAIR data guid

# Another prompt version

In [8]:
import json
import pandas as pd

def build_prompt(row):
    funding_source = str(row.get("fundingSource", "Not provided")).strip()
    institute = str(row.get("institute", "Not provided")).strip()
    is_human = str(row.get("isHumanStudy", "No")).strip()
    consent = str(row.get("consentDescription", "")).strip()
    data_collected = str(row.get("element_1A", "Not provided")).strip()

    consent_block = ""
    if is_human.lower() in ["yes", "true", "1"]:
        consent_block = f"\n- If yes, data sharing consent: {consent if consent else 'Not provided'}"

    prompt = f"""
You are a FAIR data expert helping researchers make their data more Findable, Accessible, Interoperable, and Reusable (FAIR).

Your task is to generate practical FAIR data instructions for a grant proposal being submitted to {funding_source}, specifically targeting {institute}.

Proposal context:
- Funding source: {funding_source}
- Human subjects study: {is_human}{consent_block}
- Data to be collected: {data_collected}

Based on these inputs, provide clear, practical, and user-friendly FAIR data guidance organized under the following sections:

1. Recommended file formats
Explain which file format(s) should be used for this type of data and why they are appropriate for FAIR sharing and reuse.

2. Relevant tools and software
List tools, software, or platforms that can help prepare, convert, validate, organize, document, or share this type of data.

3. Recommended dataset structure or standard
State the dataset structure, schema, or community standard that should be followed, if one exists.

4. Dataset organization details
Provide detailed guidance on how the dataset should be organized, including folder structure, file naming conventions, table organization, metadata arrangement, versioning considerations, and any other important technical details.

5. Metadata and documentation files to include
List the metadata files, README files, data dictionaries, codebooks, protocols, or other documentation that should accompany the dataset.

6. Tools for metadata preparation and validation
Suggest tools that can help create, validate, manage, or standardize metadata for this dataset.

7. De-identification and privacy considerations
State whether de-identification is required. Explain what should be done based on the subject type, the data modality, and whether the study involves human participants. If consent language affects sharing, explain that clearly.

8. Recommended repositories
Suggest appropriate repositories for sharing the data, prioritizing repositories recommended, required, or commonly accepted by the funding source and data type.

9. Recommended data licenses
Recommend appropriate data license(s), prioritizing those expected, suggested, or required by the funding source. If restrictions apply because of human subjects or consent limitations, explain that clearly.

Instructions for your response:
- Make the guidance practical, concise, and easy for researchers to follow.
- Tailor all recommendations to the combination of funding source, human-subject status, consent language, and data modality.
- When funder policies, privacy, or consent restrictions affect the answer, explain that explicitly.
- When no single standard or repository is certain, label the answer as "Recommended approach" and provide the best evidence-based guidance.
- Prefer community-accepted standards, open formats, and reusable metadata practices whenever possible.
- Keep the tone professional, helpful, and grant-writing appropriate.

Return your answer in valid JSON format only.
Do not include markdown, explanation, or extra text.

Use exactly this JSON structure:
{{
  "Recommended file formats": "",
  "Relevant tools and software": "",
  "Recommended dataset structure or standard": "",
  "Dataset organization details": "",
  "Metadata and documentation files to include": "",
  "Tools for metadata preparation and validation": "",
  "De-identification and privacy considerations": "",
  "Recommended repositories": "",
  "Recommended data licenses": ""
}}
"""
    return prompt.strip()


# Create prompt column
df["prompt"] = df.apply(build_prompt, axis=1)

# Show first prompt
print(df["prompt"].iloc[0])

You are a FAIR data expert helping researchers make their data more Findable, Accessible, Interoperable, and Reusable (FAIR).

Your task is to generate practical FAIR data instructions for a grant proposal being submitted to NIH, specifically targeting National Institute of Mental Health (NIMH).

Proposal context:
- Funding source: NIH
- Human subjects study: yes
- If yes, data sharing consent: All research participants will be consented for broad data sharing.
- Data to be collected: Demographic, clinical, and MRI, 1 H fMRS and fMRI imaging data will be acquired from 110 affected youth and 110 matched healthy controls (described in detail in sections C.3 and C.4 of this application). All data will be deidentified prior to receipt by the repository, but the information needed to generate a global unique identifier for the NIMH Data Archive (NDA) will be collected for each subject.

Based on these inputs, provide clear, practical, and user-friendly FAIR data guidance organized under the

In [10]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

def ask_gpt(prompt_text):
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {
                "role": "system",
                "content": "You are a FAIR data expert helping generate practical FAIR instructions."
            },
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        temperature=0
    )
    return response.choices[0].message.content

# Apply to each row
df["gpt_answer"] = df["prompt"].apply(ask_gpt)

# Save output
df.to_excel("FAIR_instructions_with_prompts_and_answers2.xlsx", index=False)

In [27]:
import json
import re
import pandas as pd

def parse_json_result(text):
    empty_result = {
        "Recommended file formats": "",
        "Relevant tools and software": "",
        "Recommended dataset structure or standard": "",
        "Dataset organization details": "",
        "Metadata and documentation files to include": "",
        "Tools for metadata preparation and validation": "",
        "De-identification and privacy considerations": "",
        "Recommended repositories": "",
        "Recommended data licenses": ""
    }

    try:
        if pd.isna(text):
            return pd.Series(empty_result)

        text = str(text).strip()

        # If GPT added extra text before/after JSON, extract only the JSON part
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            text = match.group(0)

        data = json.loads(text)

        return pd.Series({
            "Recommended file formats": data.get("Recommended file formats", ""),
            "Relevant tools and software": data.get("Relevant tools and software", ""),
            "Recommended dataset structure or standard": data.get("Recommended dataset structure or standard", ""),
            "Dataset organization details": data.get("Dataset organization details", ""),
            "Metadata and documentation files to include": data.get("Metadata and documentation files to include", ""),
            "Tools for metadata preparation and validation": data.get("Tools for metadata preparation and validation", ""),
            "De-identification and privacy considerations": data.get("De-identification and privacy considerations", ""),
            "Recommended repositories": data.get("Recommended repositories", ""),
            "Recommended data licenses": data.get("Recommended data licenses", "")
        })

    except Exception:
        return pd.Series(empty_result)


# Keep only your original columns first
base_cols = [
    "link",
    "id",
    "fundingSource",
    "institute",
    "isHumanStudy",
    "consentDescription",
    "element_1A",
    "prompt",
    "gpt_answer"
]

df_base = df[base_cols].copy()

# Parse JSON from gpt_answer
parsed_cols = df_base["gpt_answer"].apply(parse_json_result)

# Combine original columns + parsed columns only once
df_clean = pd.concat([df_base, parsed_cols], axis=1)

# Preview selected columns
print(df_clean[[
    "Recommended file formats",
    "Relevant tools and software",
    "Recommended dataset structure or standard"
]].head())

# Save clean file
df_clean.to_excel("FAIR_instructions_with_parsed_columns.xlsx", index=False)

                            Recommended file formats  \
0  For MRI and fMRI imaging data, use NIfTI (.nii...   
1  For raw whole genome sequencing (WGS) data, us...   
2  For single-cell omics data matrices (cells × f...   
3  For structural MRI data, use NIfTI (.nii or .n...   

                         Relevant tools and software  \
0  Use dcm2niix for converting DICOM to NIfTI/NIf...   
1  Use bcl2fastq or Illumina BaseSpace for FASTQ ...   
2  Scanpy (Python), Seurat (R), AnnData (Python),...   
3  Use MRI data conversion and validation tools s...   

           Recommended dataset structure or standard  
0  Follow the Brain Imaging Data Structure (BIDS)...  
1  Follow the NIMH Data Archive (NDA) data dictio...  
2  Follow the Minimum Information About a Single ...  
3  Follow the Brain Imaging Data Structure (BIDS)...  


In [29]:
df_clean

,link,id,fundingSource,institute,isHumanStudy,consentDescription,element_1A,prompt,gpt_answer,Recommended file formats,Relevant tools and software,Recommended dataset structure or standard,Dataset organization details,Metadata and documentation files to include,Tools for metadata preparation and validation,De-identification and privacy considerations,Recommended repositories,Recommended data licenses
0,https://grants.nih.gov/sites/default/files/flm...,1,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,"Demographic, clinical, and MRI, 1 H fMRS and f...",You are a FAIR data expert helping researchers...,"{\n ""Recommended file formats"": ""For MRI and ...","For MRI and fMRI imaging data, use NIfTI (.nii...",Use dcm2niix for converting DICOM to NIfTI/NIf...,Follow the Brain Imaging Data Structure (BIDS)...,Organize data in a BIDS-compliant folder struc...,Include a top-level README file describing the...,Use the BIDS Validator for checking BIDS compl...,De-identification is required for all human su...,Deposit all data in the NIMH Data Archive (NDA...,"For NDA, data are shared under NDA Data Use Te..."
1,https://grants.nih.gov/sites/default/files/flm...,2,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,Our genomic study will be registered with dbGa...,You are a FAIR data expert helping researchers...,"{\n ""Recommended file formats"": ""For raw whol...","For raw whole genome sequencing (WGS) data, us...",Use bcl2fastq or Illumina BaseSpace for FASTQ ...,Follow the NIMH Data Archive (NDA) data dictio...,Organize data in a hierarchical folder structu...,Include a comprehensive README.txt describing ...,Use the NDA Data Dictionary Tool and NDA Valid...,De-identification is required for all human su...,Submit raw and processed genomic data to dbGaP...,For human genomic and clinical data with broad...
2,https://grants.nih.gov/sites/default/files/flm...,3,NIH,National Institute of Mental Health (NIMH),no,no,"As detailed in the Research Strategy Section, ...",You are a FAIR data expert helping researchers...,"{\n ""Recommended file formats"": ""For single-c...",For single-cell omics data matrices (cells × f...,"Scanpy (Python), Seurat (R), AnnData (Python),...",Follow the Minimum Information About a Single ...,Organize the dataset in a top-level folder nam...,"README file describing dataset contents, struc...","MetaSRA, Annotare (EMBL-EBI), HCA Metadata Sch...",No human subjects data are included in the ini...,"For mouse single-cell omics data, deposit raw ...",Apply the Creative Commons Attribution 4.0 Int...
3,https://grants.nih.gov/sites/default/files/flm...,4,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,The data to be shared will include MRI images ...,You are a FAIR data expert helping researchers...,"{\n ""Recommended file formats"": ""For structur...","For structural MRI data, use NIfTI (.nii or .n...",Use MRI data conversion and validation tools s...,Follow the Brain Imaging Data Structure (BIDS)...,Organize MRI data in a BIDS-compliant folder s...,"Include a README file describing the dataset, ...",Use the BIDS Validator (web or command-line) t...,De-identification is required for all shared h...,Deposit all data in the NIMH Data Archive (NDA...,"Use the NDA Data Use Certification Agreement, ..."


In [30]:
print(len(df_clean.columns))
print(df_clean.columns.tolist())

18
['link', 'id', 'fundingSource', 'institute', 'isHumanStudy', 'consentDescription', 'element_1A', 'prompt', 'gpt_answer', 'Recommended file formats', 'Relevant tools and software', 'Recommended dataset structure or standard', 'Dataset organization details', 'Metadata and documentation files to include', 'Tools for metadata preparation and validation', 'De-identification and privacy considerations', 'Recommended repositories', 'Recommended data licenses']
